# TS67 Skeletonization

## Setup

In [1]:
from mcf2swc import *

from swctools import SWCModel, FrustaSet, PointSet, plot_model

import logging
logging.basicConfig(level=logging.INFO)

import os

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


## Load Mesh and skeleton

In [2]:
spine_idx = 67
mcf_qst = 0.5
mcf_mcst = 10
obj_name = f"TS{spine_idx}"
skeleton_name = f"TS{spine_idx}_qst{mcf_qst}_mcst{mcf_mcst}"
mm = MeshManager(mesh_path=f"../data/mesh/processed/{obj_name}_simplified.obj")
raw_skeleton = SkeletonGraph.from_txt(f"../data/mcf_skeletons/{skeleton_name}.polylines.txt")
raw_skeleton.prune_short_branches_inplace(10)
mm.visualize_mesh_3d(skel=raw_skeleton)

INFO:mcf2swc.mesh:Loaded mesh: 1694 vertices, 3394 faces


# Skeleton optimization

In [ ]:
from mcf2swc import (
    SkeletonGraph,
    MeshManager,
    SkeletonOptimizer,
    SkeletonOptimizerOptions,
)

do_skeleton_optimization = True

if do_skeleton_optimization:
    opts = SkeletonOptimizerOptions(
        max_iterations=50,
        step_size=1.0,
        smoothing_weight=0.1,
        verbose=True,
    )
    # Create optimizer and check for surface crossing
    optimizer = SkeletonOptimizer(raw_skeleton, mm.mesh, opts)
    has_crossing, num_outside, max_dist = optimizer.check_surface_crossing()
    print(f"Surface crossing: {has_crossing}")
    print(f"Points outside: {num_outside}/{raw_skeleton.total_points()}")
    print(f"Max distance: {max_dist:.4f}")
    # Run optimization
    print("\nOptimizing skeleton...")
    optimized_skeleton = optimizer.optimize()

    skeleton = optimized_skeleton
    all_skeletons = [raw_skeleton, optimized_skeleton]
else:
    skeleton = raw_skeleton
    all_skeletons = [raw_skeleton]


mm.visualize_mesh_3d(skel=all_skeletons, skel_color=["crimson", "blue"])


INFO:mcf2swc.skeleton_optimizer:Surface crossing detected: 34/257 nodes outside mesh (max distance: 12.4376)
INFO:mcf2swc.skeleton_optimizer:Surface crossing detected: 34/257 nodes outside mesh (max distance: 12.4376)


Surface crossing: True
Points outside: 34/257
Max distance: 12.4376

Optimizing skeleton...


INFO:mcf2swc.skeleton_optimizer:Starting skeleton optimization...
INFO:mcf2swc.skeleton_optimizer:  Nodes: 257
INFO:mcf2swc.skeleton_optimizer:  Max iterations: 50
INFO:mcf2swc.skeleton_optimizer:  Step size: 1.0000
INFO:mcf2swc.skeleton_optimizer:  Smoothing weight: 0.1000
INFO:mcf2swc.skeleton_optimizer:  Iteration 0: avg movement = 0.892687
INFO:mcf2swc.skeleton_optimizer:  Iteration 10: avg movement = 0.864682
INFO:mcf2swc.skeleton_optimizer:  Iteration 20: avg movement = 0.861842
INFO:mcf2swc.skeleton_optimizer:  Iteration 30: avg movement = 0.859471
INFO:mcf2swc.skeleton_optimizer:  Iteration 40: avg movement = 0.864793
INFO:mcf2swc.skeleton_optimizer:Optimization complete


# Fit morphology

In [4]:
spacing = 100

swc_out_dir = f"../data/swc/current/{skeleton_name}"

# check if directory exists, if not create it
if not os.path.exists(swc_out_dir):
    os.makedirs(swc_out_dir)

radius_strategy_list = [
    "equivalent_area",
    # "equivalent_perimeter",
    # "section_median",
    # "section_circle_fit",
    # "nearest_surface",
]
for radius_strategy in radius_strategy_list:
    print(f"Computing skeleton for radius_strategy={radius_strategy} ...", end="")
    morph = fit_morphology(
        mm.mesh, 
        skeleton, 
        options=FitOptions(
        spacing=spacing,
        radius_strategy=radius_strategy,
        snap_polylines_to_mesh=False)
)
    # write swc to file
    morph.to_swc_file(f"{swc_out_dir}/TS{spine_idx}_s{spacing}_{radius_strategy}.swc")
    morph.print_attributes(node_info=False, edge_info=False)
    print("DONE")

INFO:mcf2swc.graph_fitting:Tracing start: mesh[V=1694,F=3394], skeleton[nodes=238,edges=239], spacing=100, radius_strategy=equivalent_area
INFO:mcf2swc.graph_fitting:Edge group 0: input_pts=15 -> samples=3
INFO:mcf2swc.graph_fitting:Edge group 1: input_pts=16 -> samples=3
INFO:mcf2swc.graph_fitting:Edge group 2: input_pts=23 -> samples=4
INFO:mcf2swc.graph_fitting:Edge group 3: input_pts=8 -> samples=2


Computing skeleton for radius_strategy=equivalent_area ...

INFO:mcf2swc.graph_fitting:Edge group 4: input_pts=8 -> samples=2
INFO:mcf2swc.graph_fitting:Edge group 5: input_pts=3 -> samples=2
INFO:mcf2swc.graph_fitting:Edge group 6: input_pts=14 -> samples=3
INFO:mcf2swc.graph_fitting:Edge group 7: input_pts=7 -> samples=2
INFO:mcf2swc.graph_fitting:Edge group 8: input_pts=11 -> samples=3
INFO:mcf2swc.graph_fitting:Edge group 9: input_pts=8 -> samples=2
INFO:mcf2swc.graph_fitting:Edge group 10: input_pts=26 -> samples=4
INFO:mcf2swc.graph_fitting:Edge group 11: input_pts=14 -> samples=3
INFO:mcf2swc.graph_fitting:Edge group 12: input_pts=2 -> samples=2
INFO:mcf2swc.graph_fitting:Edge group 13: input_pts=4 -> samples=2
INFO:mcf2swc.graph_fitting:Edge group 14: input_pts=50 -> samples=6
INFO:mcf2swc.graph_fitting:Edge group 15: input_pts=10 -> samples=2
INFO:mcf2swc.graph_fitting:Edge group 16: input_pts=8 -> samples=2
INFO:mcf2swc.graph_fitting:Edge group 17: input_pts=22 -> samples=4
INFO:mcf2swc.graph_fitting:Edge group 18: input_pts=6 -> samp

MorphologyGraph: nodes=35, edges=36, components=1, cycles=2, branch_points=11, leaves=9, self_loops=0, density=0.0605
DONE


# Plotting

In [5]:
# plot using swctools
make_html = False

for radius_strategy in radius_strategy_list:
    swc_filepath = f"{swc_out_dir}/TS{spine_idx}_s{spacing}_{radius_strategy}.swc"
    model = SWCModel.from_swc_file(swc_filepath)
    model.print_attributes(node_info=False, edge_info=False)
    frusta = FrustaSet.from_swc_model(model)
    title = f"TS{spine_idx}_s{spacing}_{radius_strategy}"
    fig = plot_model(swc_model=model, frusta=frusta, slider=True, title=title)
    fig.show()
    if make_html:
        fig.write_html(f"../plotly/TS{spine_idx}_s{spacing}_{radius_strategy}.html")


INFO:swctools.io:parse_swc start strict=True validate_reconnections=True float_tol=1e-09
INFO:swctools.io:parse_swc done records=37 reconnections=2 header=6
INFO:swctools.model:SWCModel.from_parse_result records=37 reconnections=2 header=6
INFO:swctools.model:SWCModel.from_swc_file built nodes=37 edges=36 strict=True validate_reconnections=True
INFO:swctools.geometry:batch_frusta count=36 sides=16 end_caps=False verts=1152 faces=1152
INFO:swctools.geometry:FrustaSet.from_swc_model edges=36 sides=16 end_caps=False
INFO:swctools.geometry:batch_frusta count=36 sides=16 end_caps=False verts=1152 faces=1152
INFO:swctools.geometry:FrustaSet.scaled radius_scale=0.0
INFO:swctools.geometry:batch_frusta count=36 sides=16 end_caps=False verts=1152 faces=1152
INFO:swctools.geometry:FrustaSet.scaled radius_scale=0.05
INFO:swctools.geometry:batch_frusta count=36 sides=16 end_caps=False verts=1152 faces=1152
INFO:swctools.geometry:FrustaSet.scaled radius_scale=0.1
INFO:swctools.geometry:batch_frusta 

SWCModel: nodes=37, edges=36, components=1, cycles=0, branch_points=9, roots=1, leaves=11, self_loops=0, density=0.0541


INFO:swctools.geometry:batch_frusta count=36 sides=16 end_caps=False verts=1152 faces=1152
INFO:swctools.geometry:FrustaSet.scaled radius_scale=0.5
INFO:swctools.geometry:batch_frusta count=36 sides=16 end_caps=False verts=1152 faces=1152
INFO:swctools.geometry:FrustaSet.scaled radius_scale=0.55
INFO:swctools.geometry:batch_frusta count=36 sides=16 end_caps=False verts=1152 faces=1152
INFO:swctools.geometry:FrustaSet.scaled radius_scale=0.6
INFO:swctools.geometry:batch_frusta count=36 sides=16 end_caps=False verts=1152 faces=1152
INFO:swctools.geometry:FrustaSet.scaled radius_scale=0.65
INFO:swctools.geometry:batch_frusta count=36 sides=16 end_caps=False verts=1152 faces=1152
INFO:swctools.geometry:FrustaSet.scaled radius_scale=0.7
INFO:swctools.geometry:batch_frusta count=36 sides=16 end_caps=False verts=1152 faces=1152
INFO:swctools.geometry:FrustaSet.scaled radius_scale=0.75
INFO:swctools.geometry:batch_frusta count=36 sides=16 end_caps=False verts=1152 faces=1152
INFO:swctools.geom